# 📘 9.1 Stability of Ordinary Differential Equations
###

This notebook formalise the concept of *numerical stability*, building on the behaviour that was observed when solving a basic linear ODE using Euler Method and Runge-Kutta methods from Notebooks [7.3](https://colab.research.google.com/drive/1fGC3AhR4GCxkD0tnTFpre3Av38sCPOLZ?usp=sharing) and [7.4](https://colab.research.google.com/drive/191bRXMsLSQNT-E1j4BgNzV_6aVum6CeV?usp=sharing).

---

### 🎯 Learning Objectives
> By the end of this notebook, you should be able to:
>
> * Define numerical stability in the context of both *Linear Algebra* and *Differential Equations*.
> * Be able to evalute the *forward error* and *backwards error* of algorithms in linear algebra.
> * Understand how in the solution of differential equations, stability relates to the bounded accumulation of total error and how that error is a combination of *truncation error* and *rounding error*.
> * Recognise that while a smallest step size may ensure *stability* it may not be the optimal due to need for more steps and greater accumulation of rounding errors.
> * Be able to use perturbation theory to predict stability of ODEs.

## 📂 9.1.0 How to use this notebook

This notebook focuses on the concepts of numerical stability, how this relates to error propagation, and analytical means of determining the stability of methods for solving differential equations.
> ### 🧭 Theory preface
> You are not expected to memorise the derivations, or be able to comeup with the error propagation expressions in a test environment.
> Concentrate on understanding what error growth or decay means for stability.
> Understand the process for the pertubation based stability analysis. You should be able to use this approach to evaluate other methods of similar complexity.

## 📂 9.1.1 Types of Numerical Stability

---

 In numerical calculations, there are two primary contexts in which the definition of **numerical stability** differs. The first relates to the numerical calculation in *linear algebra*, the second, the numerical solutions of ordinary and partial differential equations.

> ### 📖 Definition
> **Numerically Stability (linear algebra):** refers to how a calculation differs from the true solution to due instabilities which are caused by proximity to singularities of various kinds, such as very small singular values. The Solution of an ill-conditioned matrix, may for example become numerically unstable.
>
> **Numerically Stability (differential equation):** refers to how errors propagate through an algorithm and affect the result. A code is *numerically unstable* when errors, like rounding errors and/or small fluctuations in input data, cause a large deviation of final answer from the exact solution.

> ### ⚠️ Point of note
> A *numerically stable* algorithm is **not** free from errors, but the errors remain bounded, rather than growing to overwhelm the results.





### 🔹 Numerical Stability in Linear Algebra

Before we get tied up in how numerical stability impacts the solution to ODEs, I think it is worth taking a moment to consider the *other* numerical stability you will encounter - i.e. that which occurs in linear algebra.

So far, we have discussed **truncation errors** and **round-off errors**, the name of which correspond to the source of the error. In linear algebra you will also encounter the terms **forward error** and **backwards error**.

To help define these, let us consider a simple linear system,
$$ \mathbf{A}\cdot \mathbf{x} = \mathbf{b},$$

\begin{eqnarray}
\mathbf{A} =\begin{bmatrix}
2 & 1\\
1 & 3
\end{bmatrix}, \mathbf{b} = \begin{bmatrix}
3\\
4
\end{bmatrix}
\end{eqnarray}

where $\mathbf{x}$ is the solution for which we wish to solve, with an exact solution

$$\mathbf{x} = \begin{bmatrix} 1\\1 \end{bmatrix},$$

when $\mathbf{b}$ is the input.

Let's say that in solving $\mathbf{x}$, our algorithm returns the solution
$$\hat{\mathbf{x}} = \begin{bmatrix} 1.01\\ 0.99\end{bmatrix},$$
owing to a combination of *round-off* and *truncation error*.

In this case the solution has returned an error
$$\hat{\mathbf{x}}-\mathbf{x}=\begin{bmatrix}1.01\\0.99\end{bmatrix}-\begin{bmatrix}1\\1\end{bmatrix}=\begin{bmatrix}0.01\\-0.01\end{bmatrix}.$$

Given the error here can vary between data points, we can reduces this to a single value by considering the RMSE or more formally the Euclidean norm:

$$\|\hat{\mathbf{x}}-\mathbf{x}\|=\sqrt{(0.01)^2+(-0.01)^2}=0.0141$$

which you will often find referred to as the **forward error**.

> ### 📖 Definition
> **forward error** is defined as the difference between the computed solution, $\hat{\mathbf{x}}$ and the true solution $\mathbf{x}$.

This raises the question, what input is $\hat{\mathbf{x}}$ the solution of?

We can calculate the corresponding input $\hat{\mathbf{b}}$ as
$$\hat{\mathbf{b}}=\mathbf{A}\cdot\hat{\mathbf{x}} =
\begin{bmatrix}
2 & 1\\1 & 3
\end{bmatrix}\begin{bmatrix}
1.01\\0.99
\end{bmatrix} =\begin{bmatrix}
3.01\\3.98
\end{bmatrix} $$

and define a residual $\mathbf{r}$ which is the difference between $\hat{\mathbf{b}}$ and $\mathbf{b}$

$$ \mathbf{r} = \hat{\mathbf{b}}-\mathbf{b}=\begin{bmatrix}
3.01\\3.98
\end{bmatrix} - \begin{bmatrix}
3\\4
\end{bmatrix}= \begin{bmatrix}
0.01\\-0.02
\end{bmatrix}$$

This residual corresponds to the smallest pertubation to $\mathbf{b}$, which makes $\hat{\mathbf{x}}$ and exact solution, which is referred to as the **backward error**.

> ### 📖 Definition
> **backward error** is defined as smallest perturbation to the input  that makes the computed solution exact for a nearby problem (e.g. exact for $\hat{\mathbf{x}}$ rather than exact for $\mathbf{x}$).

Graphically, as shown in Figure 9.1, we can present the **forward error** as distance between the solution we compute by solving $\mathbf{A}\cdot\mathbf{x}=\mathbf{b}$, $\hat{\mathbf{x}}$ and the true solution, with a **backward error** computed from direct evaluation of our system $\mathbf{A}\hat{\mathbf{x}}=\hat{\mathbf{b}}$.

<figure>
  <img src="https://drive.google.com/uc?id=1CC58mddcJ33L8mkti5HX59-eOwaqI3f6" alt="" style="width:100%;max-width:400px;">
  <figcaption style="text-align: center; font-style: italic; margin-top: 5px;">
    Figure 9.1: Diagram showing the Forward Error and the Backward Error, and their relation to the exact solution and the input that would lead to the calculated solution.
  </figcaption>
</figure>

At first glance, it might seem like we would only be interested in the **forward error**, i.e. how far is my calculation away from the true answer. However, it is not uncommon for an ill-conditioned matrix $\mathbf{A}$ to result in a large **forward error**, but maintain a relatively small **backward error**.

*Why is this the case?*

An **ill-conditioned** matrix might result in a large forward error due to the sensitivity of the system, not because of *round-off* or *truncation errors* in the algorithm.

> ### ⚠️ Point of note
> A large **forward error** is not necessarily the fault of the algorithm! It may be due to the nature of the system.

Another benefit practical benefit of the **backward error** is that we can compute a backwards error even when we don't know the true solution.

Consider the example above, if we didn't know that the true solution was
$$\mathbf{x} = \begin{bmatrix} 1\\1 \end{bmatrix},$$
we could not evaluate the **forward error**,
$$\hat{\mathbf{x}}-\mathbf{x}$$

yet we can still compute the **backward error**
$$\hat{\mathbf{b}}-\mathbf{b}=\mathbf{A}\hat{\mathbf{x}}-\mathbf{b}$$

> ### ⚠️ Point of note
> **Backward Error** can be calculated even when we don't know the true solution.
> This is effectively just solving for $\hat{\mathbf{x}}$ and seeing how well this solution satisfies the linear equation. *So **backward error** is like a sanity check for your calculation!*

---

*So how does this relate to stability?*

---

Given this distinction between errors, the *stability* in the case of linear algebra an algorithm can be,

> ### 📖 Definition
> **Forward Stable**, when the relative **forward error** is of the order of the product of the condition number and the machine precision, $u$,
$$ \frac{\|\hat{\mathbf{x}}-\mathbf{x}\|}{\|\mathbf{x}\|} \approx \mathcal{O}(\kappa\cdot u),$$
where
$$\kappa = \text{cond}(\mathbf{A})=\frac{\sigma_{max}(\mathbf{A})}{\sigma_{min}(\mathbf{A})}$$
is the condition of the matrix as given by the ratio of the largest and smallest singular values (see [Notebook 2.3](https://colab.research.google.com/drive/1Tbk9niqTk9PsVL3NnK4WciqxFcSL6zeg?usp=playground)).

or

> ### 📖 Definition
> **Backward Stable**, when the realtive **backward error** is of the order of machine precision, $u$.
$$ \frac{\| \hat{\mathbf{b}}-\mathbf{b} \| }{\|\mathbf{b}\|} \approx \mathcal{O}(u)$$

> ### ⚠️ Point of note
> While not getting into the mathematics, it is a property of the condition of the matrix that a linear algorithm that is **backward stable** is automatically **forwards stable**, but the converse is not true.

For these reasons, it is common practice to consider the **backward stability** of a linear algorithm.

#### 🧪 Example Forward and Backward Stability of Numpy.linalg

Consider the example we had above and evaluate the errors and stability, i.e. consider the system
$$ \mathbf{A}\cdot \mathbf{x} = \mathbf{b},$$
where
\begin{eqnarray}
\mathbf{A} =\begin{bmatrix}
2 & 1\\
1 & 3
\end{bmatrix}, \mathbf{b} = \begin{bmatrix}
3\\
4
\end{bmatrix}
\end{eqnarray}

In [ ]:
import numpy as np
from numpy import linalg as la
# define matrix and input
A = np.array([[2,1],[1,3]])
b = np.array([3,4])

# determine condition
kappa = la.cond(A)
print(f'Condition number = {kappa}')

# solve for x
x = la.solve(A,b)
print(f'calculated x = {x}')

print(f'Forward error = {la.norm(x-np.array([1,1]))}')
print(f'Backward error = {la.norm(A@x-b)}')

Condition number = 2.6180339887498953
x = [1. 1.]
Forward error = 0.0
Backward error = 0.0


In this case, the solution is exact, and both forward and backward errors are less than machine precision.

What happens if our system is ill-conditioned?

Consider a classic ill-conditioned matrix,

$$ \mathbf{A}\cdot \mathbf{x} = \mathbf{b},$$
where
\begin{eqnarray}
\mathbf{A} =\begin{bmatrix}
1 & 1\\
1 & 1.0001
\end{bmatrix}, \mathbf{b} = \begin{bmatrix}
2\\
2.0001
\end{bmatrix}
\end{eqnarray}
where the true solution is $\mathbf{x} = [1,1]^T$.

In [ ]:
import numpy as np
from numpy import linalg as la
# define matrix and true solution
A = np.array([[1,1],[1,1.0001]],dtype=np.float32)
xt = np.array([1,1],dtype=np.float32)
b = np.array([2,2.0001],dtype=np.float32)
# determine condition
kappa = la.cond(A)
print(f'Condition number = {kappa}')
print(f'float32 machine precision = {np.finfo(np.float32).resolution}')

# solve for x
x = la.solve(A,b)
print(f'calculated x = {x}')

print(f'Relative Forward error = {la.norm(x-np.array([1,1]))/la.norm(np.array([1,1]))}')
print(f'Relative Backward error = {la.norm(A@x-b)/la.norm(b)}')

print(f'Compared to cond(A)*machine balance = {kappa*np.finfo(np.float32).resolution}')

Condition number = 39995.3671875
float32 machine precision = 9.999999974752427e-07
calculated x = [1.0011919 0.9988081]
Relative Forward error = 0.001191884279623693
Relative Backward error = 0.0
Compared to cond(A)*machine balance = 0.03999536857008934


In this case the **backward error** is zero, but the **forward error** is no longer neglibible. The algorithm is clearly **backward stable**. In considering the **forward stability**, we need to account for the *condition number* of the matrix $\mathbf{A}$.

When comparing the **forward error** with the product of the condition number and the machine balance, we would still say that the `np.linalg` algorithm used here is **numerically stable**.

### 🔹 Numerical Stability in Differential Equations (ODEs-PDEs)

Differential equations are different in that we generally require a solution to march forward in time or space, and converge towards a solution. In doing so we not only have to discretise our equation, which bring with it a **truncation error**, but we also have to select a step size. This raise the possibility that errors will compound and grow to a point where they mask the true solution or, as is often the case, will grow in an unbounded fashion that will cause value to reach the limit of the data types which have been tasked with storing the solution.

*Remember the precision of a floating-point number if finite - so as the magnitude of our solution grows, the absolute uncertainty increases!*

Methods for solving a *differential equation* are said to be **stable** if they produce a bounded solution which imitates the exact solution. If the solution explodes, or diverges from the exact solution, it is **unstable**.

If a method is stable for all values of the parameter, it is said to be absolutely or unconditionally stable. If it is stable for some values of the parameter, it is said to be conditionally stable.

> ### 📖 Definition
> **Absolutely or unconditionally stable** refers to a methods that is stable for all values of the parameters, e.g. for all step sizes.
>
> **Conditionally stable** refers to a methods that is stable for some, but not all values of the parameters, e.g. stable for a finite range of step sizes.

Consider what you have already seen in terms of the stability of Euler's methods when applied to a first order ODE.

## 📂 9.1.2 Solution of ODEs by Euler's Method and Numerical Stability

---

So far, we have looked at the solution of Ordinary Differential Equations (ODEs) from initial values, incrementing our solution in time or space via Euler's method or multi-stage Runge-Kutta schemes.

In [Notebook 7.3](https://colab.research.google.com/drive/1fGC3AhR4GCxkD0tnTFpre3Av38sCPOLZ?usp=sharing) we presented the stability of the Euler Method in terms of an amplification factor, *A*, based on the stepsize, *h*, and the coefficient $\lambda$, when solving a first-order ODE of the form
$$\frac{\mbox{d}y}{\mbox{d}t} = -\lambda y$$
where
$$ A = 1-\lambda h.$$

Stable only when

$$
|A|<1
\quad\Longleftrightarrow\quad
0<h\lambda<2.
$$

Let's us take a moment to consider what is happending here in more detail and why this is the case.

---
Why is the method becoming unstable?

---

Remember, the Euler Method can be derived from the Taylor series expansion about a point $i$,

$$y_{i+1} = y_i+y'_i(t_{i+1}-t_i) + \frac{y''_i}{2}(t_{i+1}-t_i)^2 + ...$$

where if we define $h$ as the interval between point $i$ and $i+1$, and consider only the first-order term we arrive at Euler's Method,

$$y_{i+1} = y_i+y'_i h + \epsilon_t,\tag{9.1}$$

where for small interval, the truncation error, $\epsilon_t \approx \frac{f'_i}{2}h^2$, or $\epsilon_t \approx \mathcal{O}(h^2)$.

> ### ⚠️ Point of note
> This is a **local truncation error** meaning that each step from at point $i$ to $i+1$ will have an error $\mathcal{O}(h^2)$.

### 🔹 Consider the source of error in each step of the Euler method

If we are using this to solve
$$\frac{\mbox{d}y}{\mbox{d}t} = -\lambda y$$

then this ODE tells us that at any point $i$ we can express the local gradient $y'_i$ as
$$y'_i = \frac{\mbox{d}y}{\mbox{d}t} = -\lambda y_i$$

Hence, any $i+1$ we can solve for $y$ but subsituting this into Equation 9.1,
\begin{align}
y_{i+1} &= y_i-\lambda y_i h + \epsilon_t\\
y_{i+1} &= y_i(1-\lambda h) + \epsilon_t. \tag{9.2}
\end{align}

So far we have only considered the **truncation error**, but assuming that $y$ is a float, we also have to consider a **rounding error**, $\epsilon_r$, such that  

$$y_{i+1} = y_i(1-\lambda h) + \epsilon_t + \epsilon_r. \tag{9.3}$$

Following this process we can express $y_{i+2}$ as
\begin{align}
y_{i+2} &= y_{i+1}(1-\lambda h) + \epsilon_t +\epsilon_{r,i+2}\\
y_{i+2} &= \left(y_i(1-\lambda h) + \epsilon_t + \epsilon_{r,i+1}\right)(1-\lambda h)+ \epsilon_t +\epsilon_{r,i+2}\\
y_{i+2} &= y_i(1-\lambda h)^2 +\epsilon_t\left[1+(1-\lambda h)\right]+(1-\lambda h)\epsilon_{r,i+1}+\epsilon_{r,i+2} \tag{9.4}
\end{align}

The subscripts $i+1$, $i+2$ are added because the rounding-error is not a constant, but depends on the magnitude of $y$, i.e. $\epsilon_{i+1} = u|y_{i+1}|$.

### 🔹 Consider accumulation of error across multiple steps of the Euler method

Given, we are concerned with the error at the $n$-th point, it becomes tedious to trace the error from $i=1$ to $i=n$, but we can observe some patterns here.

For instance, the solution for $y_n$ that will be returned by the method appears in the first terms of Equation 9.4. Each step will involve a further product of $(1-\lambda h)$, hence we can express the solution as
$$y_n = (1-\lambda h)^n y_0,\tag{9.5}$$
where $y_0$ is the initial value of $y(t=0)$.

The next term in Equation 9.4 is the accumulated *truncation error*, in which each step adds an error $\epsilon_t\approx \mathcal{O}(h^2)$, as the trunction error from the previous steps is scaled by a factor $(1-\lambda h)$ for each subsequent step. Hence the *truncation error* for $i+1$ can be expressed as
$$\epsilon_{t,i+1} = (1-\lambda h)\epsilon_{t,i} + \epsilon_t$$

For $n$ step this can be expressed as
$$\epsilon_{t,n} = \sum^{n-1}_{i=0} (1-\lambda h)^{n-1-i}\epsilon_t\tag{9.6}$$

A similar pattern exists for the **rounding error**
$$ \epsilon_{r,i+1} = (1-\lambda h)\epsilon_{r,i} + \rho_{i+1},$$
where
$$ \rho_{i+1} = u|y_{i+1}|,$$
is the additional error resulting from the finite precision of the floating-point value and $u$ is the machine precision. Since $y_{i+1} = (1-\lambda h)y_i$ in the present case,
$$ \rho_{i+1} = u|y_{i+1}|= u|(1-\lambda h)y_i|$$

where accumulated over $n$ steps, with an initial *rounding error*, $r_0$,
For $n$ step this can be expressed as
$$\epsilon_{r,n} = (1-\lambda h)^n \epsilon_{r,0}+\sum^{n-1}_{i=0} (1-\lambda h)^{n-1-i}\rho_i\tag{9.7},$$
where
$$\rho_i = u|(1-\lambda h)^{i+1} y_0|.$$

From both Equations 9.6 and 9.7, we can see that both the truncations and rounding errors will grow if $(1-\lambda h) > 1$. Consistent with our previously defined amplification factor.

> ### ⚠️ Point of note
> If either truncation or rounding errors grow with each step, the simulation can be come unstable.

### 🔹 Selecting the Optimal Step Size - Is it just about stabilty?

In this case, if $h < 1/\lambda$, the truncation error terms will decay as shown by Equation 9.6. Similarly, even though there are more steps required and a greater source of round-off error, Equation 9.7 shows that these two will be damped, and the solution will steadily converge.

*So we should choose the smallest step size to ensure stability?*

A small step size, $h→0$, will reduce the truncation error in each step as truncation error is $\approx \mathcal{O}(h^2)$, but the number of steps required from $t=0$ to some finite time $t=\Delta t$ is $n=\Delta t/h$ so a smaller step size means more steps.

Consider what happens to the truncation error as $h→0$, where $n\rightarrow \infty$ such that $n-1 \approx \infty$. From Equation 9.6,

$$\epsilon_{t,n} = \sum^{\infty}_{i=0} (1-\lambda h)^{\infty-i}\epsilon_t\tag{9.6}$$

since $\infty-i$ ranges from infinity to zero as $n$ ranges from zero to infinity, we can flip this series if we let $j=\infty -1$, and express the series as

$$\epsilon_{t,n\rightarrow \infty} \approx \epsilon_t\sum^{\infty}_{j=0} (1-\lambda h)^{j}$$

Since $\sum^\infty_0 q^j = \frac{1}{1-q}$, we can express the residual as
$$\epsilon_{t,n\rightarrow \infty}\approx \epsilon_t \frac{1}{\lambda h}.$$

Given we know that the order of the truncation, i.e. $\epsilon_t \approx \mathcal{O}(h^2)\approx Ch^2$, where $C$ is a constant, we can express the truncation error in the limit of small $h$ and infinite steps as

\begin{align}
\epsilon_{t,n\rightarrow \infty}&\approx \frac{C}{\lambda}h\\
\epsilon_{t,n\rightarrow \infty}&\approx C_1h,\end{align}

such that the **truncation error** is order $\mathcal{O}(h)$ and shrinks with step size.

So far the small step size seems like the winner. However, we also need to consider the **rounding error**. From Equation 9.7, having subbed in $\rho_i$ and representing the machine precision as $u$ that that $\epsilon_{r,0} = u|y_0|$,

\begin{align}
\epsilon_{r,n} &= (1-\lambda h)^n u|y_0| +\sum^{n-1}_{i=0} (1-\lambda h)^{n-1-i}u(1-\lambda h)^{i+1} |y_0|\\
\epsilon_{r,n} &= u|y_0|(1-\lambda h)^n  +u|y_0|\sum^{n-1}_{i=0} (1-\lambda h)^n
\end{align}

The sum is no longer with respect to $i$ and can be evaluated as $\sum^{n-1}_i= n$ such that the accumulated rounding error becomes

\begin{align}
\epsilon_{r,n} &= u|y_0|(1-\lambda h)^n  +u|y_0|n (1-\lambda h)^n\\
\epsilon_{r,n} &= (1+n)u|y_0|(1-\lambda h)^n
\end{align}

For a fixed time interval $\Delta t$, the number of step is related to the step size as $n=\Delta t/h$ and in the limit as $h\rightarrow 0$, $|1-\lambda h| \ll 1$ hence

\begin{align}
\epsilon_{r,n\rightarrow \infty} &\approx (1+\frac{\Delta t}{h})u|y_0|\\
\epsilon_{r,n\rightarrow \infty} &\approx \Delta t|y_0|\frac{u}{h}\\
\epsilon_{r,n\rightarrow \infty} &\approx C_2\frac{u}{h}
\end{align}

The **total error** is the combination of the two, i.e.
$$ \epsilon_{total} \approx C_1 h + C_2\frac{u}{h}$$

If $h$ is large, the **truncation error** will dominate. If $h$ is small the **rounding error** will grow as more and more steps are required.

The minimum error occurs when $d\epsilon/dh = 0$,

$$\frac{\mbox{d}\epsilon_{total}}{\mbox{d}h} = C_1 - C_2\frac{u}{h^2}=0$$
$$ h_{opt} \approx \sqrt{C_3 u}$$

> ### ⚠️ Point of note
> The optimal step size is not the minimum step size, as rounding error begins to dominate at small step sizes. The optimal step size is $\propto \sqrt{u}$ where $u$ is the floating point machine precision.



## 📂 9.1.3 A General Means of Determining The Stability of Differential Solution Methods

---

While we can determine stability by calculating the propagation of truncation and random error, this quickly becomes a tedious process, even for a relatively simple case like this ODE and Euler Method.

Fortunately, we don't need to understand the individual error terms in order to determine stability. Instead we can determine stability by looking at how a method responds to a **perturbation**. This perturbation represents the noise, without needing to know the exact noise level.

For instance, return to our example ODE,
$$\frac{\mbox{d}y}{\mbox{d}t} = -\lambda y$$
where the basic Euler Method was shown to yield the solution
$$y_n = (1-\lambda h)^n y_0,\tag{9.5}$$
where $y_0$ is the initial value of $y(t=0)$.

Let's now introduce a complex *perturbation*, $\delta_0$ to our initial solution, such that or initial value becomes $\tilde{y}_0 = y_0 +\delta_0$, where $\delta_0 = re^{i\theta}$. The resulting solution after $n$ iterations becomes

\begin{align}
\tilde{y}_n &= (1-\lambda h)^n\tilde{y}\\
\tilde{y}_n &= (1-\lambda h)^n(y_0+\delta_0)\tag{9.7}\\
\end{align}

Subtrating the Equation 9.5 from 9.7 yields the difference between the solution and the perturbed solution, i.e. $\tilde{y}_n-y_n = \delta_n$, where

$$\delta_n = (1-\lambda h)^n \delta_0.$$

For linear ODEs, the perturbation will have the same amplification factors as the solution, but this method can also be applied in more complex cases.

If we let $1-\lambda h$ be complex, $1-\lambda h = \rho e^{i\theta}$ then,

\begin{align}
\delta_n &= (\rho e^{i\theta})^n \delta_0\\
\delta_n &= \rho^n e^{in\theta} \delta_0
\end{align}

From the definition of the complex number the amplitude of the change in the perturbation is
$$|\delta_n| = \rho^n|\delta_0|$$
and the phase shift is
$$\phi(\delta_n) = n\theta$$

The pertubation will not grow as long as $\rho \le 1$. Given that by definition
$$\rho = |1-\lambda h|$$
the method will be stable as long as
$$|1-\lambda h| \le 1.$$

If we let $\lambda = \mbox{Re}+i\mbox{Im}$ in term of its real and imaginary components, assuming $h$ is real then

\begin{align}
|1-(\mbox{Re}+i\mbox{Im})h| \le 1\\
|(1-\mbox{Re} h) + (\mbox{Im} h)| \le 1
\end{align}
The magnitude of the complex number $(1-\mbox{Re} h) + \mbox{Im}$ being
$$\sqrt{(1-\mbox{Re} h)^2 + (\mbox{Im} h)^2} \le 1 $$

This corresponds to a circle in the complex plane as depicted in Figure 9.2. If $\lambda$ is real then the Euler Method will be stable for values $-2 > \lambda h > 0$, if complex, the value must remain in the unit circle.

<figure>
  <img src="https://drive.google.com/uc?id=1I6ZevDZx4RMRdPrpCpPjIGMRI0nyuf71" alt="" style="width:100%;max-width:400px;">
  <figcaption style="text-align: center; font-style: italic; margin-top: 5px;">
    Figure 9.2: Stability plane for Euler Method Solution of dy/dt = -λy.
  </figcaption>
</figure>


## Summary of Stability Terms

| Term | Definition |
| :------------------------------ | :--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Numerically Stability (linear algebra)** | Refers to how a calculation differs from the true solution due to instabilities which are caused <br>by proximity to singularities of various kinds, such as very small singular values.<br>The solution of an ill-conditioned matrix, may for example become numerically unstable. |
| **Numerically Stability (differential equation)** | Refers to how errors propagate through an algorithm and affect the result.<br>A code is *numerically unstable* when errors, like rounding errors and/or small fluctuations in input data,<br>cause a large deviation of final answer from the exact solution. |
| **Forward Stable** | When the relative **forward error** is of the order of the product of the condition number<br>and the machine precision, $u$. |
| **Backward Stable** | When the relative **backward error** is of the order of machine precision, $u$. |
| **Absolutely or unconditionally stable** | Refers to a method that is stable for all values of the parameters,<br>e.g., for all step sizes. |
| **Conditionally stable** | Refers to a method that is stable for some, but not all values of the parameters,<br>e.g., stable for a finite range of step sizes. |

## 🔗 Bridge to Notebook 9.2

This notebook began with a general definition of Numerical Stability and the distinction between how the term **stability** is used in the context of **linear algebra** and how it is used in the context of solving **differential equations**. This formalised concepts of **forward** and **backwards** errors, which you encounter in solving matrices in Week 2.

Using the example of Euler's Method, we then demonstrated how the growth of **truncation errors** and **random errors** occur, what this means in terms of stability and why the **optimal step size** is not necessarily the smallest step size.

Finally, we demonstrated how a perturbation can be used to determine the stability plane for an algorithm, without needing to know the source of the error or quantify error growth rates.

Notebook 9.2 will discuss STIFF ODEs, why they present a different challenge regarding numerical stability, and ways we can solve them.

#🤖 AI Acknowledgment

MS CoPilot v25.6.440430001 was used proof read this notebook and to generate example matrices to demonstrate a well and ill-condition forward and backwards errors, based on prompts provided by the author. The cases were then modified by the author to provide consistent presentation.